from sagemaker import get_execution_role
role = get_execution_role()
print(role)  # should be arn:aws:iam::...:role/...

In [14]:
# Cell 1: Clone or pull repository (SSH - no token needed!)
from pathlib import Path
%cd /mnt/custom-file-systems/s3/shared/
REPO_NAME = "ocr-agentic-rag"

if Path(REPO_NAME).exists():
    print(f"✓ Repository exists - pulling latest changes...")
    %cd {REPO_NAME}
    !git pull origin master
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone git@github.com:leemingloon/ocr-agentic-rag.git
    %cd {REPO_NAME}

print("\n✓ Repository ready")
!git status

/mnt/custom-file-systems/s3/shared
Cloning ocr-agentic-rag...
Cloning into 'ocr-agentic-rag'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 100 (delta 12), reused 97 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 568.12 KiB | 164.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
Updating files: 100% (69/69), done.
/mnt/custom-file-systems/s3/shared/ocr-agentic-rag

✓ Repository ready
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


In [25]:
import boto3
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()                      # your SageMaker role
account_id = boto3.client('sts').get_caller_identity()['Account']
region = sess.boto_region_name                   # should be ap-southeast-2
ecr_repo_name = "ocr-agentic-rag-processing"     # create this repo in ECR if not exists
image_tag = "latest"

full_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:{image_tag}"

# Create ECR repo if missing
!aws ecr create-repository --repository-name {ecr_repo_name} --region {region} || echo "Repo exists"

# Docker login to ECR
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

# Build (this is the long part — 10–25 min first time)
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Clean any old failed build cache first (helps space)
!docker builder prune -f

# Build with the required network flag
!docker build --network sagemaker -t {full_image_uri} .

# Push
!docker push {full_image_uri}

print("Image pushed:", full_image_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix

An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'ocr-agentic-rag-processing' already exists in the registry with id '200915316557'
Repo exists
WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded
/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
DEPRECATED: The legacy builder i

In [26]:
# Check if the image is valid
!docker images | grep ocr-agentic-rag-processing

200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   latest    01e07e77a67b   4 minutes ago       4.44GB
200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   <none>    0f0aa83ca3b7   About an hour ago   4.44GB
200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   <none>    375f1fa980ae   2 hours ago         4.44GB
200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   <none>    d916a25f1268   3 hours ago         4.44GB
200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing   <none>    1f066b990d07   4 hours ago         4.44GB


In [27]:
import sagemaker
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput

role = sagemaker.get_execution_role()
bucket_name = "sagemaker-ocr-rag-mingloon"
full_image_uri = "200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing:latest"

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [28]:
from sagemaker.processing import Processor

processor = Processor(
    role=role,
    image_uri=full_image_uri,
    instance_count=1,
    instance_type="ml.t3.medium",           # cheap & sufficient
    max_runtime_in_seconds=3600,            # 1 hour max safety cap
    sagemaker_session=sagemaker.Session()
)

input_s3 = f"s3://{bucket_name}/input/credit_risk/uci_credit_default/"  # points to where your CSV is
output_s3 = f"s3://{bucket_name}/output/"

processor.run(
    inputs=[
        ProcessingInput(
            source=input_s3,
            destination="/opt/ml/processing/input",
            input_name="input_data"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=output_s3,
            output_name="results"
        )
    ],
    arguments=[
        "--mode", "sagemaker",
        "--s3-bucket", bucket_name,
        "--dry-run"  # dry-run remove this line later for full run
    ],
    wait=True,
    logs=True
)

print("Job completed. Check outputs in S3:", output_s3)
print("Job name (for console):", processor._current_job_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
...............⚠ PaddleOCR not available
ℹ️  DRY-RUN mode → continuing without Anthropic key
🧪 DRY-RUN MODE: No API calls will be made (testing pipeline only)
SageMaker Evaluation with S3 Integration
✓ S3 client initialized: s3://sagemaker-ocr-rag-mingloon
⚠ Downloading reference data from S3: s3://sagemaker-ocr-rag-mingloon/data/reference/training_data.csv
✓ Reference data downloaded and cached
✓ Reference data loaded: 30000 samples, 25 features
1. Loading reference data from S3...
✓ Reference data loaded: 30000 samples
2. Running credit risk evaluation...
⚠ Downloading FinBERT from HuggingFace: ProsusAI/finbert
#015Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]#015Loading weights:   0%|          | 1/201 [00:00<00:00, 6626.07it/s, Materializing param=ber

In [18]:
import boto3

# Get your current session region automatically (best practice)
session = boto3.session.Session()
region = session.region_name  # This will be 'ap-southeast-1' for you

bucket_name = "sagemaker-ocr-rag-mingloon"  # Keep or change to something unique like sagemaker-ocr-rag-mingloon-2026

s3_client = boto3.client('s3', region_name=region)  # Explicit region for safety

try:
    # For us-east-1 you can omit CreateBucketConfiguration entirely
    # But since you're likely NOT in us-east-1, include it
    create_config = {}
    if region != 'us-east-1':
        create_config = {'CreateBucketConfiguration': {'LocationConstraint': region}}

    s3_client.create_bucket(
        Bucket=bucket_name,
        **create_config
    )
    print(f"Bucket created successfully in region {region}: {bucket_name}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists and is owned by you: {bucket_name}")
except s3_client.exceptions.BucketAlreadyExists:
    print(f"Bucket name {bucket_name} is already taken globally — try a different name (add random suffix like -v2 or -20260216)")
except Exception as e:
    print("Error:", e)
    print("Full exception info for debug:", str(e))

Bucket created successfully in region ap-southeast-2: sagemaker-ocr-rag-mingloon


In [21]:
# Check if bucket exists
!aws s3 ls s3://{bucket_name}/ || echo "Bucket empty or access issue"

In [31]:
# Setup Kaggle credentials from AWS Secrets Manager (requires IAM: KaggleAPIKeyAccess)
import os
import json

def _load_kaggle_token():
    # 1. AWS Secrets Manager (secret name: KAGGLE_API_KEY)
    try:
        import boto3
        from botocore.exceptions import ClientError
        client = boto3.client("secretsmanager")
        response = client.get_secret_value(SecretId="KAGGLE_API_KEY")
        raw = response.get("SecretString", "").strip()
        if raw.startswith("{"):
            data = json.loads(raw)
            token = data.get("KAGGLE_API_KEY") or data.get("KAGGLE_API_TOKEN") or data.get("api_key") or next(iter(data.values()), None)
        else:
            token = raw
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
            print("✓ Kaggle token loaded from AWS Secrets Manager (KAGGLE_API_KEY)")
            return
    except Exception as e:
        print(f"Secrets Manager (KAGGLE_API_KEY): {e}")

    # 2. Fallback: environment variable (local/dev or already set)
    token = os.environ.get("KAGGLE_API_KEY") or os.environ.get("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        print("✓ Kaggle token from environment")
        return
    print("⚠ KAGGLE_API_TOKEN not set: add KAGGLE_API_KEY in Secrets Manager or set env var")

_load_kaggle_token()

# Quick verification (first 10 chars only)
!echo $KAGGLE_API_TOKEN | head -c 10 && echo "... (token is set)"

KGAT_4da7c... (token is set)


In [32]:
# Tests if Kaggle API authentication is working (lists your datasets or public ones)
!kaggle datasets list -m  # -m shows your models if any, or just public if no models

No datasets found


In [25]:
# pip install kaggle (first-run only)
!pip install kaggle

In [27]:
# Downloads evaluation datasets, saves to data/credit_risk/{dataset_name}/ (CSVs, JSONs, texts)
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

bucket_name = "sagemaker-ocr-rag-mingloon"  # Your existing bucket

!bash scripts/download_credit_datasets.sh --mode sagemaker --s3-bucket {bucket_name} --tier 1

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
Credit Risk Dataset Downloader

Mode: --mode

✓ Created directory structure

Tier 1 Datasets (Must-Have - All FREE)

1. Lending Club (2.9M loans)
   Source: https://www.kaggle.com/datasets/wordsforthewise/lending-club
   Purpose: PD model training
   Download: Requires Kaggle API
   Command: kaggle datasets download -d wordsforthewise/lending-club

2. FiQA Sentiment (1,173 samples)
   Source: https://huggingface.co/datasets/financial_phrasebank
   Purpose: NLP sentiment validation
   Download: Via HuggingFace datasets library

3. FinanceBench (10,231 Q&A)
   Source: https://huggingface.co/datasets/PatronusAI/financebench
   Purpose: Risk memo Q&A validation
   Download: Via HuggingFace datasets library

4. ECTSum (2,425 summaries)
   Source: https://github.com/rajdeep345/ECTSum
   Purpose: Risk memo summarization
   Download: git clone https://github.com/rajdeep345/ECTSum.git

5. Credit Card Default UCI (30K)
   Source: https://archive

In [10]:
import boto3

s3 = boto3.client("s3")
s3.upload_file("evaluation_results_credit_risk.json", "sagemaker-ocr-rag-mingloon", "risk_memos/test.json")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 import boto3                                                                                 │
│   2                                                                                              │
│   3 s3 = boto3.client("s3")                                                                      │
│ ❱ 4 s3.upload_file("evaluation_results_credit_risk.json", "sagemaker-ocr-rag-mingloon", "ris     │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/boto3/s3/inject.py:175 in upload_file                    │
│                                                                                                  │
│   172 │   │   transfer.                                                                          │
│   173 │   """                                                                                    │
│   174 │   with S3Transfer(self, Config) as transfer:                                             │
│ ❱ 175 │   │   return transfer.upload_file(                                                       │
│   176 │   │   │   filename=Filename,                                                             │
│   177 │   │   │   bucket=Bucket,                                                                 │
│   178 │   │   │   key=Key,                                                                       │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/boto3/s3/transfer.py:373 in upload_file                  │
│                                                                                                  │
│   370 │   │   │   filename, bucket, key, extra_args, subscribers                                 │
│   371 │   │   )                                                                                  │
│   372 │   │   try:                                                                               │
│ ❱ 373 │   │   │   future.result()                                                                │
│   374 │   │   # If a client error was raised, add the backwards compatibility layer              │
│   375 │   │   # that raises a S3UploadFailedError. These specific errors were only               │
│   376 │   │   # ever thrown for upload_parts but now can be thrown for any related               │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/s3transfer/futures.

In [69]:
Confirm the ECR repository exists and has the image
!aws ecr describe-repositories --repository-names ocr-agentic-rag-processing --region ap-southeast-2 || echo "Repo not found"
!aws ecr describe-images --repository-name ocr-agentic-rag-processing --region ap-southeast-2


An error occurred (AccessDeniedException) when calling the DescribeRepositories operation: User: arn:aws:sts::200915316557:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker is not authorized to perform: ecr:DescribeRepositories on resource: arn:aws:ecr:ap-southeast-2:200915316557:repository/ocr-agentic-rag-processing because no identity-based policy allows the ecr:DescribeRepositories action
Repo not found

An error occurred (RepositoryNotFoundException) when calling the DescribeImages operation: The repository with name 'ocr-agentic-rag-processing' does not exist in the registry with id '200915316557'


In [70]:
# Create the ECR repository properly
repo_name = "ocr-agentic-rag-processing"
region = "ap-southeast-2"

!aws ecr create-repository --repository-name {repo_name} --region {region} --output json

{
    "repository": {
        "repositoryArn": "arn:aws:ecr:ap-southeast-2:200915316557:repository/ocr-agentic-rag-processing",
        "registryId": "200915316557",
        "repositoryName": "ocr-agentic-rag-processing",
        "repositoryUri": "200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing",
        "createdAt": "2026-02-15T23:09:34.252000+00:00",
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": false
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}


In [76]:
# Re-build and push the image
account_id = "200915316557"
region = "ap-southeast-2"
repo_name = "ocr-agentic-rag-processing"
tag = "latest"

full_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:{tag}"

# Login
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

# Build with network flag (required in Studio)
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag
!docker build --network sagemaker -t {full_image_uri} .

# Push
!docker push {full_image_uri}

print("Image URI for job:", full_image_uri)

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded
/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  10.24MB
Step 1/11 : FROM python:3.11-slim
 ---> 466c0182639b
Step 2/11 : RUN apt-get update && apt-get install -y     git     tesseract-ocr     libgl1     libglib2.0-0     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 569178f19693
Step 3/11 : WORKDIR /app
 ---> Using cache
 ---> 85f957a64cfb
Step 4/11 : COPY . .
 ---> Using cache
 ---> b01e77cddbed
Step 5/11 : RUN pip install --no-cache-dir     torch torchvision --index-url https://downloa

In [77]:
# Verify push worked
!aws ecr describe-images --repository-name {repo_name} --region {region}

{
    "imageDetails": [
        {
            "registryId": "200915316557",
            "repositoryName": "ocr-agentic-rag-processing",
            "imageDigest": "sha256:7479301f4ecfc85374aaeab999ebedba2b4a59052fe6f303680bc82769d8cda7",
            "imageTags": [
                "latest"
            ],
            "imageSizeInBytes": 1605886176,
            "imagePushedAt": "2026-02-15T23:29:05.984000+00:00",
            "imageManifestMediaType": "application/vnd.docker.distribution.manifest.v2+json",
            "artifactMediaType": "application/vnd.docker.container.image.v1+json",
            "imageStatus": "ACTIVE"
        }
    ]
}


In [72]:
# Confirm repo & image exist (now with permissions)
!aws ecr describe-repositories --repository-names {repo_name} --region {region}
!aws ecr describe-images --repository-name {repo_name} --region {region}

{
    "repositories": [
        {
            "repositoryArn": "arn:aws:ecr:ap-southeast-2:200915316557:repository/ocr-agentic-rag-processing",
            "registryId": "200915316557",
            "repositoryName": "ocr-agentic-rag-processing",
            "repositoryUri": "200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing",
            "createdAt": "2026-02-15T23:09:34.252000+00:00",
            "imageTagMutability": "MUTABLE",
            "imageScanningConfiguration": {
                "scanOnPush": false
            },
            "encryptionConfiguration": {
                "encryptionType": "AES256"
            }
        }
    ]
}
{
    "imageDetails": []
}


In [73]:
account_id = "200915316557"
region = "ap-southeast-2"
repo_name = "ocr-agentic-rag-processing"
tag = "latest"

full_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:{tag}"

!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [74]:
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Clean old build cache to save space
!docker builder prune -f

# Build with required --network flag
!docker build --network sagemaker -t {full_image_uri} .

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

request returned Forbidden for API route and version http://%2Fdocker%2Fproxy.sock/v1.44/build/prune?filters=&keep-storage=0, check if the server supports the requested API version
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  10.24MB
Step 1/11 : FROM python:3.11-slim
 ---> 466c0182639b
Step 2/11 : RUN apt-get update && apt-get install -y     git     tesseract-ocr     libgl1     libglib2.0-0     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 569178f19693
Step 3/11 : WORKDIR /app
 ---> Using cache
 ---> 85f957a64cfb
Step 4/1

In [75]:
!docker push {full_image_uri}

The push refers to repository [200915316557.dkr.ecr.ap-southeast-2.amazonaws.com/ocr-agentic-rag-processing]

7cb3c67c: Preparing 
6326f458: Preparing 
01a4de2b: Preparing 
f2178a17: Preparing 
030165c5: Preparing 
ea4cac7d: Preparing 
8e8d19a2: Preparing 
efb4ec4c: Preparing 
denied: User: arn:aws:sts::200915316557:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker is not authorized to perform: ecr:InitiateLayerUpload on resource: arn:aws:ecr:ap-southeast-2:200915316557:repository/ocr-agentic-rag-processing because no identity-based policy allows the ecr:InitiateLayerUpload action


In [ ]:
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Clean old build cache to save space
!docker builder prune -f

# Build with required --network flag
!docker build --network sagemaker -t {full_image_uri} .

In [48]:
# 1. Check if the bucket exists and you can list it
!aws s3 ls s3://sagemaker-ocr-rag-mingloon/ || echo "Cannot list bucket — check IAM role"

# 2. List everything recursively (look for any files)
!aws s3 ls s3://sagemaker-ocr-rag-mingloon/ --recursive | head -20

# 3. Specifically check the expected input prefix
!aws s3 ls s3://sagemaker-ocr-rag-mingloon/input/credit_risk/ --recursive || echo "Prefix is empty or inaccessible"

Prefix is empty or inaccessible


In [49]:
!pip install datasets huggingface_hub gitpython

In [50]:
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

bucket_name = "sagemaker-ocr-rag-mingloon"

# Clean local data dir to avoid confusion
!rm -rf data/credit_risk

# Run the script again (it should handle Kaggle/HF now that env is set)
!bash scripts/download_credit_datasets.sh --mode sagemaker --s3-bucket {bucket_name} --tier 1

# If script fails, do manual sync of whatever was downloaded locally
!aws s3 sync data/credit_risk/ s3://{bucket_name}/input/credit_risk/ --exclude "*" --include "*.csv" --include "*.json" --include "*.txt" --include "*.pdf" --include "*.jpg" --include "*.png"

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
Credit Risk Dataset Downloader

Mode: --mode

✓ Created directory structure

Tier 1 Datasets (Must-Have - All FREE)

1. Lending Club (2.9M loans)
   Source: https://www.kaggle.com/datasets/wordsforthewise/lending-club
   Purpose: PD model training
   Download: Requires Kaggle API
   Command: kaggle datasets download -d wordsforthewise/lending-club

2. FiQA Sentiment (1,173 samples)
   Source: https://huggingface.co/datasets/financial_phrasebank
   Purpose: NLP sentiment validation
   Download: Via HuggingFace datasets library

3. FinanceBench (10,231 Q&A)
   Source: https://huggingface.co/datasets/PatronusAI/financebench
   Purpose: Risk memo Q&A validation
   Download: Via HuggingFace datasets library

4. ECTSum (2,425 summaries)
   Source: https://github.com/rajdeep345/ECTSum
   Purpose: Risk memo summarization
   Download: git clone https://github.com/rajdeep345/ECTSum.git

5. Credit Card Default UCI (30K)
   Source: https://archive

In [51]:
!aws s3 ls s3://{bucket_name}/input/credit_risk/ --recursive | head -15
!aws s3 ls s3://{bucket_name}/input/credit_risk/ --summarize


Total Objects: 0
   Total Size: 0


In [52]:
!aws s3 cp --dryrun README.md s3://sagemaker-ocr-rag-mingloon/test-upload/README.md

(dryrun) upload: ./README.md to s3://sagemaker-ocr-rag-mingloon/test-upload/README.md


In [53]:
# Create local data dir if missing
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag
!mkdir -p data/credit_risk/uci_credit_default

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag


In [54]:
# Download UCI Credit Card Default dataset (small CSV, ~30K rows, direct link)
!wget -O data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv \
    https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls

--2026-02-15 22:46:47--  https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv’

data/credit_risk/uc     [      <=>           ]   5.28M  3.96MB/s    in 1.3s    

2026-02-15 22:46:49 (3.96 MB/s) - ‘data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv’ saved [5539328]



In [55]:
# Convert XLS to CSV if needed (UCI provides XLS, but script expects CSV)
!pip install pandas openpyxl
import pandas as pd

xls_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.xls"
csv_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv"

df = pd.read_excel(xls_file, header=1)  # skip first row header
df.to_csv(csv_file, index=False)
print("Converted to CSV:", csv_file)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 xls_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.xls"         │
│    6 csv_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv"         │
│    7                                                                                             │
│ ❱  8 df = pd.read_excel(xls_file, header=1)  # skip first row header                             │
│    9 df.to_csv(csv_file, index=False)                                                            │
│   10 print("Converted to CSV:", csv_file)                                                        │
│   11                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/pandas/io/excel/_base.py:495 in read_excel               │
│                                                                                                  │
│    492 │                                                                                         │
│    493 │   if not isinstance(io, ExcelFile):                                                     │
│    494 │   │   should_close = True                                                               │
│ ❱  495 │   │   io = ExcelFile(                                                                   │
│    496 │   │   │   io,                                                                           │
│    497 │   │   │   storage_options=storage_options,                                              │
│    498 │   │   │   engine=engine,                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/pandas/io/excel/_base.py:1550 in __init__                │
│                                                                                                  │
│   1547 │   │   │   if xlrd_version is not None and isinstance(path_or_buffer, xlrd.Book):        │
│   1548 │   │   │   │   ext = "xls"                                                               │
│   1549 │   │   │   else:                                                                         │
│ ❱ 1550 │   │   │   │   ext = inspect_excel_format(                                               │
│   1551 │   │   │   │   │   content_or_path=path_or_buffer, storage_options=storage_options       │
│   1552 │   │   │   │   )                                                                         │
│   1553 │   │   │   │   if ext is None:                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/pandas/io/excel/_base.py:1402 in inspect_excel_format    │
│                                                                                                  │
│   1399 │   if isinstance(content_or_path, bytes):                                                │
│   1400 │   │   content_or_path = BytesIO(content_or_path)                                        │
│   1401 │                                                                                         │
│ ❱ 1402 │   with get_handle(                                                                      │
│   1403 │   │   content_or_path, "rb", storage_options=storage_options, is_text=False             │
│   1404 │   ) as handle:                                                                          │
│   1405 │   │   stream = handle.handle                      

In [56]:
import requests
import os

%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Create dir if missing
os.makedirs("data/credit_risk/uci_credit_default", exist_ok=True)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
local_path = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.xls"

print(f"Downloading from {url}...")

response = requests.get(url, stream=True)
response.raise_for_status()  # Raise error if download fails

with open(local_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"Downloaded successfully: {local_path}")
print(f"File size: {os.path.getsize(local_path) / (1024 * 1024):.2f} MB")

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
Downloaded successfully: data/credit_risk/uci_credit_default/default_of_credit_card_clients.xls
File size: 5.28 MB


In [58]:
# Install xlrd (required for .xls files)
!pip install xlrd

In [59]:
!pip install pandas openpyxl  # already done, but safe to rerun

import pandas as pd

xls_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.xls"
csv_file = "data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv"

df = pd.read_excel(xls_file, header=1)  # UCI has header on row 2
df.to_csv(csv_file, index=False)

print("Converted to CSV:", csv_file)
print("First few rows:")
print(df.head())

Converted to CSV: data/credit_risk/uci_credit_default/default_of_credit_card_clients.csv
First few rows:
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     3

In [67]:
bucket_name = "sagemaker-ocr-rag-mingloon"

!aws s3 sync data/credit_risk/ s3://{bucket_name}/input/credit_risk/ \
    --exclude "*" \
    --include "*.csv" \
    --include "*.json" \
    --include "*.txt" \
    --include "*.pdf" \
    --include "*.jpg" \
    --include "*.png"

# Verify
!aws s3 ls s3://{bucket_name}/input/credit_risk/uci_credit_default/ --recursive
!aws s3 ls s3://{bucket_name}/input/credit_risk/uci_credit_default/ --summarize

2026-02-15 22:51:03    2867208 input/credit_risk/uci_credit_default/default_of_credit_card_clients.csv
2026-02-15 22:51:03    2867208 default_of_credit_card_clients.csv

Total Objects: 1
   Total Size: 2867208


In [42]:
# Check disk usage on root/home
!df -h

# Detailed usage in home (look for large dirs like .cache, .local)
!du -sh /home/sagemaker-user/* | sort -hr | head -15

# Specifically check Poetry-related caches
!du -sh /home/sagemaker-user/.cache/pypoetry || echo "No cache dir yet"
!du -sh /home/sagemaker-user/.local || echo "No .local dir"

Filesystem      Size  Used Avail Use% Mounted on
overlay          37G  4.5M   37G   1% /
tmpfs            64M     0   64M   0% /dev
tmpfs           1.9G     0  1.9G   0% /sys/fs/cgroup
shm             394M   48K  394M   1% /dev/shm
tmpfs           1.9G  744K  1.9G   1% /docker
/dev/nvme1n1     16G   16G  155M 100% /home/sagemaker-user
/dev/nvme0n1p1  180G   19G  162G  11% /etc/resolv.conf
tmpfs           1.9G     0  1.9G   0% /proc/acpi
tmpfs           1.9G     0  1.9G   0% /sys/firmware
s3fs             64P     0   64P   0% /mnt/custom-file-systems/s3/shared
4.0K	/home/sagemaker-user/README.md
0	/home/sagemaker-user/shared
16G	/home/sagemaker-user/.cache/pypoetry
6.4M	/home/sagemaker-user/.local


In [40]:
# Run end-to-end on various modes
!python run_e2e.py --mode sagemaker --s3-bucket {bucket_name} --eval --dry-run

Traceback (most recent call last):
  File "/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/run_e2e.py", line 48, in <module>
    from examples.full_e2e_demo import run_full_e2e_demo
  File "/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/examples/full_e2e_demo.py", line 26, in <module>
    import cv2
ModuleNotFoundError: No module named 'cv2'


In [34]:
from sagemaker import get_execution_role; print(get_execution_role())

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
arn:aws:iam::200915316557:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [9]:
# Install Poetry first
!pip install poetry -q

# Install project dependencies
!poetry install

print("✓ Dependencies installed")

Installing dependencies from lock file

Package operations: 3 installs, 0 updates, 0 removals

  - Installing paddlepaddle (3.3.0): Pending...
  - Installing torch (2.10.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 0%
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 10%
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 18%
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 20%
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 27%
  - Installing triton (3.6.0): Pending...
  - Installing triton (3.6.0): Pending...
  - Installing torch (2.10.0): Downloading... 30%
  - Installi

In [8]:
# Pull latest changes from GitHub
import os
# os.chdir('/home/sagemaker-user/ocr-agentic-rag')  # Adjust path if different

!git pull origin master

print("✓ Latest code pulled from GitHub")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 533 bytes | 0 bytes/s, done.
From https://github.com/leemingloon/ocr-agentic-rag
 * branch            master     -> FETCH_HEAD
   0dcc70d..df1b4f6  master     -> origin/master
Updating 0dcc70d..df1b4f6
Fast-forward
 poetry.lock    | 2 +-
 pyproject.toml | 3 ++-
 2 files changed, 3 insertions(+), 2 deletions(-)
✓ Latest code pulled from GitHub


In [14]:
# Force install PyTorch in the Poetry environment
!poetry run pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Verify
!poetry run python -c "import torch; print(f'✓ PyTorch {torch.__version__} installed')"

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 MB 151.8 MB/s  0:00:010:00:0100:01
ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/sagemaker-user/.cache/pypoetry/virtualenvs/ocr-agentic-rag-BYNRYxNr-py3.11/lib/python3.11/site-packages/torch/__init__.py", line 430, in <module>
    _load_global_deps()
  File "/home/sagemaker-user/.cache/pypoetry/virtualenvs/ocr-agentic-rag-BYNRYxNr-py3.11/lib/python3.11/site-packages/torch/__init__.py", line 388, in _load_global_deps
    _preload_cuda_deps(err)
  File "/home/sagemaker-user/.cache/pypoetry/virtualenvs/ocr-agentic-rag-BYNRYxNr-py3.11/lib/python3.11/site-packages/torch/__init__.py", line 344, in _preload_cuda_deps
    raise err
  File "/home/sagemaker-user/.cache/pypoetry/virtualenvs/ocr-agentic-rag-BYNRYxNr-py3.11/lib/python3.11/site-packages/torch/

In [16]:
# Check disk usage
!df -h

# Clean up pip cache
!pip cache purge

# Clean up Poetry cache
!poetry cache clear pypi --all

# Remove downloaded models (they'll re-download when needed)
!rm -rf ~/.cache/huggingface/hub/*

# Check space again
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay          37G   36M   37G   1% /
tmpfs            64M     0   64M   0% /dev
tmpfs           1.9G     0  1.9G   0% /sys/fs/cgroup
shm             394M   44K  394M   1% /dev/shm
tmpfs           1.9G  744K  1.9G   1% /docker
/dev/nvme0n1p1  180G   19G  162G  11% /etc/resolv.conf
/dev/nvme1n1     16G   16G  224K 100% /home/sagemaker-user
tmpfs           1.9G     0  1.9G   0% /proc/acpi
tmpfs           1.9G     0  1.9G   0% /sys/firmware
s3fs             64P     0   64P   0% /mnt/custom-file-systems/s3/shared
Files removed: 144
No cache entries for pypi
Filesystem      Size  Used Avail Use% Mounted on
overlay          37G   37M   37G   1% /
tmpfs            64M     0   64M   0% /dev
tmpfs           1.9G     0  1.9G   0% /sys/fs/cgroup
shm             394M   44K  394M   1% /dev/shm
tmpfs           1.9G  744K  1.9G   1% /docker
/dev/nvme0n1p1  180G   19G  162G  11% /etc/resolv.conf
/dev/nvme1n1     16G   16G  189M  99% /home/sagemaker-us

In [17]:
# Work from S3-mounted directory (unlimited space)
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag

# Install packages to system (not user home)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu --no-cache-dir
!pip install sentence-transformers FlagEmbedding transformers anthropic boto3 python-dotenv --no-cache-dir

# Set environment
import os
os.environ['DRY_RUN_MODE'] = 'true'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Navigate there
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag

# Run from S3 directory
!python run_e2e.py --mode sagemaker --s3-bucket ocbc-credit-risk-demo-ap-southeast-2 --dry-run

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag
Looking in indexes: https://download.pytorch.org/whl/cpu
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 579.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 565.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 431.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 313.1 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=flagembedding-1.3.5-py3-none-any.whl size=233819 sha256=401d6734e5b892104f80e4607e25ac46c14dd9c964ba52cd346bfda6703a7af7
  Stored in directory: /tmp/pip-ephem-wheel-cache-_nomcwot/wheels/fc/1c/66/c9c846a8f8cbd9574db8d76b0a61410a087bc07d53682a54f4
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18996 sha256=c21e717c38bd8d56f1db72a3e283b92ccdfcc7ea38a7552

In [23]:
# Navigate there
%cd /mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag

# Install OpenCV
!pip install opencv-python --no-cache-dir

# Install llama-index
!pip install llama-index llama-index-core --no-cache-dir

# Downgrade NumPy to 1.x for compatibility
!pip install "numpy<2" --force-reinstall --no-cache-dir

# Reinstall faiss
!pip install faiss-cpu --force-reinstall --no-cache-dir

# Run from S3 directory
!python run_e2e.py --mode sagemaker --s3-bucket ocbc-credit-risk-demo-ap-southeast-2 --dry-run

/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 344.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.4.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-common 1.4.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 23.0.0 which is incompatible.
autogluon-multimodal 1.4.0 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.57.6 which is incompatible.
autogluon-timeseries 1.4.0 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.57.6 which is incompatible.
awswrangler 3.15.0 requires pyarrow<23.0.0,>=8.0.0, but you have

In [12]:
# Create S3 bucket in the correct region
import boto3

# Auto-detect your current region
session = boto3.session.Session()
region = session.region_name

s3 = boto3.client('s3')
bucket_name = f'ocbc-credit-risk-demo-{region}'  # Make it unique per region

print(f"Creating bucket in region: {region}")

try:
    if region == 'us-east-1':
        # us-east-1 doesn't need LocationConstraint
        s3.create_bucket(Bucket=bucket_name)
    else:
        # All other regions need LocationConstraint
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"✓ Created bucket: {bucket_name}")
except Exception as e:
    if 'BucketAlreadyExists' in str(e) or 'BucketAlreadyOwnedByYou' in str(e):
        print(f"✓ Bucket {bucket_name} already exists")
    else:
        print(f"Error: {e}")

# Save bucket name for later use
print(f"\n✓ Your S3 bucket: {bucket_name}")
print(f"✓ Region: {region}")

Creating bucket in region: ap-southeast-2
✓ Created bucket: ocbc-credit-risk-demo-ap-southeast-2

✓ Your S3 bucket: ocbc-credit-risk-demo-ap-southeast-2
✓ Region: ap-southeast-2


In [13]:
# Run pipeline in SageMaker mode (dry-run first)
!poetry run python run_e2e.py --mode sagemaker --s3-bucket ocbc-credit-risk-demo --dry-run

Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.
⚠ PaddleOCR detector unavailable in router: If you use `@root_validator` with pre=False (the default) you MUST specify `skip_on_failure=True`. Note that `@root_validator` is deprecated and should be replaced with `@model_validator`.

For further information visit https://errors.pydantic.dev/2.12/u/root-validator-pre-skip
⚠ PaddleOCR detector unavailable: PDX has already been initialized. Reinitialization is not supported.
Traceback (most recent call last):
  File "/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag/run_e2e.py", line 48, in <module>
    from examples.full_e2e_demo import run_full_e2e_demo
  File "/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag/examples/full_e2e_demo.py", line 36, in <module>
    from rag_system.chunking import DocumentChunker
  File "/mnt/custom-file-systems/s3/shared/ocr-agentic-rag/ocr-agentic-rag/rag_sy

In [ ]:
# Check if it completed successfully
import subprocess

result = subprocess.run(
    ['poetry', 'run', 'python', 'run_e2e.py', '--dry-run'],
    capture_output=True,
    text=True
)

# Show last 50 lines of output
print("="*70)
print("LAST 50 LINES OF OUTPUT:")
print("="*70)
print('\n'.join(result.stdout.split('\n')[-50:]))

# Check if successful
if "E2E Demo Complete!" in result.stdout:
    print("\n✅ SUCCESS! Pipeline ran successfully in SageMaker!")
else:
    print("\n⚠ Check output above for errors")

In [ ]:
# Remove dry-run mode to use real API calls
os.environ['DRY_RUN_MODE'] = 'false'

# Run with real API (costs ~$0.003)
!poetry run python run_e2e.py